In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [20]:
# ── 1단계: 데이터 불러오기 ──
freq = pd.read_csv("freMTPL2freq.csv")
print(f"계약 수: {len(freq):,}")
print(f"총 사고 건수: {freq.ClaimNb.sum():,}")
print(f"총 가입 기간(년): {freq.Exposure.sum():,.0f}")
print(f"단순 계산 1년당 사고율 = 총 사고 ÷ 총 기간 = "
      f"{freq.ClaimNb.sum() / freq.Exposure.sum():.4f}")

계약 수: 678,013
총 사고 건수: 36,102
총 가입 기간(년): 358,499
단순 계산 1년당 사고율 = 총 사고 ÷ 총 기간 = 0.1007


In [21]:
# ---------------------------------------------------------------- 1. 타깃 ClaimNb 점검
# 사고 5건 이상 9건 중 7건은 같은 차가 여러 계약 번호로 쪼개 기록된 것으로 보임
# 행을 지우면 "위험하다"는 정보까지 사라지므로, 삭제하지 않고 4로 cap
raw_claimnb = freq["ClaimNb"].copy()   # 시각화에서 cap 전후 비교용
n_over = (freq.ClaimNb > 4).sum()
freq["ClaimNb"] = freq["ClaimNb"].clip(upper=4)
print(f"[1] ClaimNb 4에서 cap: {n_over}건 수정")

[1] ClaimNb 4에서 cap: 9건 수정


In [22]:
# ---------------------------------------------------------------- 2. Exposure 점검
# 1년 넘는 계약(대부분 1.01~1.04)은 기록 오차 → 1로 cap
# 분모가 커지면 1년당 사고율이 작게 계산됨 (예: 1 ÷ 2.01 = 0.50)
# 짧은 계약은 진짜 값이므로 그대로 둠 → offset이 알아서 보정
raw_exposure = freq["Exposure"].copy()   # 시각화에서 cap 전후 비교용
n_over = (freq.Exposure > 1).sum()
freq["Exposure"] = freq["Exposure"].clip(upper=1)
print(f"[2] Exposure 1에서 cap: {n_over:,}건 수정")

[2] Exposure 1에서 cap: 1,224건 수정


In [23]:
# ---------------------------------------------------------------- 3. 열 고르기
# ClaimNb = 타깃, Exposure = offset (심도와 반대로 버리지 않음!)
# Area는 Density를 6구간으로 자른 것 → 둘 다 넣으면 배수가 쪼개지고 SE가 커짐
freq = freq.drop(columns=["Area"])
print("[3] Area 제거, Exposure는 offset용으로 유지")

[3] Area 제거, Exposure는 offset용으로 유지


In [24]:
# ---------------------------------------------------------------- 4. train/test 분할
# 사고 0건이 95% → ClaimNb로 층화해서 드문 3·4건 계약도 고르게 나눔
# 한 행 = 계약 1건이라 IDpol이 겹칠 일이 없으므로 그룹 분할은 필요 없음
train, test = train_test_split(freq, test_size=0.2, random_state=42,
                               stratify=freq["ClaimNb"])
train, test = train.copy(), test.copy()
print(f"[4] 분할: train {len(train):,} / test {len(test):,}")

[4] 분할: train 542,410 / test 135,603


In [25]:
# ---------------------------------------------------------------- 5. train에서 기준 학습 → 변환 적용
# 희귀 범주는 행 수가 아니라 "사고 수"로 판단
# (사고 45건인 지역은 5건만 더 나도 사고율이 11% 흔들림)
def frequent_levels(d, col, min_claims=200):
    claims = d.groupby(col)["ClaimNb"].sum()
    return set(claims[claims >= min_claims].index)


params = {"keep_region": frequent_levels(train, "Region"),
          "keep_brand": frequent_levels(train, "VehBrand")}
print(f"[5] Other로 합칠 Region: {sorted(set(train.Region) - params['keep_region'])}")
print(f"    Other로 합칠 VehBrand: {sorted(set(train.VehBrand) - params['keep_brand'])}")

AGE_BINS = [17, 21, 25, 30, 40, 50, 60, 70, 80, 101]


def preprocess(d, p):
    d = d.copy()
    d["VehAge"] = d["VehAge"].clip(upper=20)
    d["BonusMalus"] = d["BonusMalus"].clip(upper=150)
    d["DrivAgeBin"] = pd.cut(d["DrivAge"], bins=AGE_BINS).astype(str)
    d["VehPower"] = d["VehPower"].clip(upper=12).astype(str)
    d["logDensity"] = np.log(d["Density"])
    d["Region"] = d["Region"].where(d["Region"].isin(p["keep_region"]), "Other")
    d["VehBrand"] = d["VehBrand"].where(d["VehBrand"].isin(p["keep_brand"]), "Other")
    return d


train = preprocess(train, params)
test = preprocess(test, params)

[5] Other로 합칠 Region: ['Alsace', 'Auvergne', 'Champagne-Ardenne', 'Corse', 'Franche-Comte']
    Other로 합칠 VehBrand: ['B14']


In [26]:
# ---------------------------------------------------------------- 6. 최종 점검
# 실수가 있어도 모델은 조용히 돌아가므로, 규칙이 지켜졌는지 assert로 바로 멈춰서 확인
cols = ["ClaimNb", "Exposure", "DrivAgeBin", "VehAge", "BonusMalus", "VehPower",
        "VehGas", "VehBrand", "Region", "logDensity"]
assert freq["IDpol"].is_unique, "IDpol 중복"
for name, d in [("train", train), ("test", test)]:
    assert d[cols].isna().sum().sum() == 0, f"{name}에 결측치"
    assert d["ClaimNb"].max() <= 4, f"{name} ClaimNb cap 누락"
    # Exposure = 0이면 offset log(0) = -inf → numpy는 경고만 내고 계속 계산함
    assert d["Exposure"].between(0, 1, inclusive="right").all(), f"{name} Exposure가 0 이하 또는 1 초과"
for c in ["Region", "VehBrand"]:
    assert "Other" in set(train[c]), f"train {c}에 Other가 없어서 처음 보는 수준을 예측할 수 없음"
for c in ["DrivAgeBin", "VehPower", "VehBrand", "Region"]:
    assert set(test[c]) <= set(train[c]), f"test {c}에 train에 없는 수준"
print(f"[6] 점검 통과 ✅  1년당 사고율 train {train.ClaimNb.sum() / train.Exposure.sum():.4f}"
      f" / test {test.ClaimNb.sum() / test.Exposure.sum():.4f}")

keep = ["IDpol"] + cols
train[keep].to_csv("freq_train_prep.csv", index=False)
test[keep].to_csv("freq_test_prep.csv", index=False)
print("\n저장: freq_train_prep.csv, freq_test_prep.csv")


[6] 점검 통과 ✅  1년당 사고율 train 0.1006 / test 0.1007

저장: freq_train_prep.csv, freq_test_prep.csv


In [32]:
# Colab: 전처리 셀을 먼저 실행한 뒤, 이 내용을 새 셀에 통째로 붙여 넣고 실행하세요.
# ================================================================ 시각화
# 전처리에서 한 일을 그림으로 확인 (그림 3장: 화면에 보여 주고 png로도 저장)
# 위의 전처리 코드는 건드리지 않고, 결과(freq, train, test, params)만 읽어서 그림
# Mac · Windows · Colab 어디서 실행해도 한글이 나오도록 폰트를 자동으로 맞춤
import glob
import os
import subprocess
import sys

import matplotlib.pyplot as plt
from matplotlib import font_manager


# ---------------------------------------------------------------- 한글 폰트 자동 설정
def in_colab():
    # 노트북 셀에서 실행해도, !python 파일.py로 실행해도 Colab인지 알아냄
    return "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ


def register_font_file():
    """폰트 '파일'을 찾아 matplotlib에 직접 등록
    (Colab은 폰트를 설치해도 matplotlib이 실행 중에는 새 폰트를 모름 → 직접 알려 주면 재시작 불필요)"""
    patterns = ["/usr/share/fonts/**/NanumGothic.ttf",            # Linux · Colab (fonts-nanum)
                "/usr/share/fonts/**/NotoSansCJK-Regular.ttc",    # Linux (fonts-noto-cjk)
                "C:/Windows/Fonts/malgun.ttf",                    # Windows
                "/System/Library/Fonts/AppleSDGothicNeo.ttc"]     # macOS
    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            font_manager.fontManager.addfont(path)
            return font_manager.FontProperties(fname=path).get_name()
    return None


def set_korean_font():
    # 1순위: matplotlib이 이미 알고 있는 한글 폰트
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in ["Apple SD Gothic Neo", "AppleGothic", "Malgun Gothic",
                 "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]:
        if name in installed:
            return name
    # 2순위: 컴퓨터에 파일은 있는데 matplotlib이 모르는 폰트
    name = register_font_file()
    if name:
        return name
    # 3순위: Colab이면 나눔 폰트를 설치한 뒤 다시 등록
    if in_colab():
        print("[시각화] Colab에 한글 폰트가 없어서 나눔 폰트를 설치하는 중... (10~30초)")
        subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null 2>&1", shell=True)
        name = register_font_file()
        if name is None:   # 설치 목록이 오래돼서 실패한 경우 → 목록을 새로 받고 한 번 더
            subprocess.run("apt-get -qq update > /dev/null 2>&1 && "
                           "apt-get -qq install -y fonts-nanum > /dev/null 2>&1", shell=True)
            name = register_font_file()
        if name:
            return name
    print("⚠️ 한글 폰트를 찾지 못해서 그림의 한글이 □로 보일 수 있어요.")
    return None


font = set_korean_font()
if font:
    plt.rcParams["font.family"] = font
print(f"\n[시각화] 사용 폰트: {font}")

# ---------------------------------------------------------------- 공통 스타일 (심도 viz_prep.py와 같은 색)
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#898781"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "axes.unicode_minus": False,   # 한글 폰트에서 마이너스(−)가 □로 깨지는 것 방지
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK2, "ytick.color": INK2, "axes.axisbelow": True,
    "figure.facecolor": "white", "axes.facecolor": "white",
})


def save_and_show(fig, filename):
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.show()   # Colab · Jupyter에서는 셀 아래에 그림이 바로 보임


# cap 전 원본과 Other 병합 전 범주가 필요해서 원본을 다시 읽음 (전처리 코드를 고치지 않기 위해)
# 위에서 DATA 경로를 정했으면 그걸 쓰고, 없으면(Colab에 CSV를 바로 올린 경우) 현재 폴더에서 읽음
raw = pd.read_csv(globals().get("DATA", "") + "freMTPL2freq.csv")
raw_claimnb, raw_exposure = raw["ClaimNb"], raw["Exposure"]
# train_test_split은 원래 행 번호(index)를 유지하므로 같은 행의 원래 Region, VehBrand를 찾을 수 있음
train_raw = train.assign(Region=raw.loc[train.index, "Region"],
                         VehBrand=raw.loc[train.index, "VehBrand"])


def rate(d):
    """1년당 사고율 = 사고 수 ÷ 가입 기간 (빈도에서는 행 평균이 아니라 이걸 봐야 함)"""
    return d["ClaimNb"].sum() / d["Exposure"].sum()


overall = rate(train)

# ---------------------------------------------------------------- 1. 타깃과 Exposure: cap 전후 (0~2단계)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# 왼쪽: 사고 0건 계약을 빼면 사고율이 얼마나 부풀려지나 (0단계)
ax = axes[0]
pos = raw_claimnb > 0
vals = [raw_claimnb.sum() / raw_exposure.sum(), raw_claimnb[pos].sum() / raw_exposure[pos].sum()]
ax.bar([0, 1], vals, color=[BLUE, GRAY], width=0.6)
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", color=INK2)
ax.set_xticks([0, 1], ["전체 계약\n(0건 포함)", "사고 난 계약만\n(0건 제외)"])
ax.set_title("[0단계] 0건 계약을 빼면? 1년당 사고율", loc="left")
ax.set_ylabel("1년당 사고율")

# 가운데: ClaimNb 분포, cap 전 vs 후 (1단계) — 0건이 너무 많아서 y축 로그
ax = axes[1]
before = raw_claimnb.value_counts().sort_index()
after = freq["ClaimNb"].value_counts().sort_index()
ax.bar(before.index - 0.2, before.values, width=0.4, color=GRAY, label="cap 전")
ax.bar(after.index + 0.2, after.values, width=0.4, color=BLUE, label="cap 후 (4에서 자름)")
ax.set_yscale("log")
ax.set_xticks(before.index)
ax.set_title("[1단계] 계약당 사고 건수 ClaimNb (y축 로그)", loc="left")
ax.set_xlabel("사고 건수")
ax.set_ylabel("계약 수")
ax.legend(frameon=False)

# 오른쪽: Exposure 분포와 cap 위치 (2단계)
ax = axes[2]
ax.hist(raw_exposure, bins=np.arange(0, 2.05, 0.05), color=BLUE, edgecolor="white", linewidth=0.5)
ax.axvline(1, color=ORANGE, linewidth=2)
ax.text(1.05, ax.get_ylim()[1] * 0.9, f"cap = 1년\n넘는 계약 {(raw_exposure > 1).sum():,}건",
        color=INK2, fontsize=9, va="top")
ax.set_title("[2단계] 가입 기간 Exposure (cap 전)", loc="left")
ax.set_xlabel("가입 기간 (년)")
ax.set_ylabel("계약 수")

save_and_show(fig, "freq_viz_1_target.png")


# ---------------------------------------------------------------- 2. 숫자 변수: train vs test 분포 + 1년당 사고율 (4·5단계)
num_cols = [("VehAge", "차 나이 (20에서 자름)", np.arange(-0.5, 21.5, 1)),
            ("BonusMalus", "BonusMalus (150에서 자름)", np.arange(50, 152, 4)),
            ("logDensity", "log(인구 밀도)", np.linspace(0, 10.5, 22))]

fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for top, bottom, (col, title, b) in zip(axes[0], axes[1], num_cols):
    # 위: train과 test는 건수가 달라서 비율(density)로 맞춰 비교 → 층화 분할이 잘 됐나?
    top.hist(train[col], bins=b, density=True, histtype="step", linewidth=2, color=BLUE, label="train")
    top.hist(test[col], bins=b, density=True, histtype="step", linewidth=2, color=ORANGE, label="test")
    top.set_title(title, loc="left")
    top.set_ylabel("비율")

    # 아래: 구간별 1년당 사고율 (train) → 이 변수가 사고와 관련 있나?
    g = (train.assign(bin=pd.cut(train[col], b))
         .groupby("bin", observed=True)[["ClaimNb", "Exposure"]].sum())
    g = g[g["Exposure"] >= 100]   # 가입 기간이 너무 적은 구간은 흔들려서 제외
    mid = [iv.mid for iv in g.index]
    bottom.plot(mid, g["ClaimNb"] / g["Exposure"], color=BLUE, marker="o", markersize=3)
    bottom.axhline(overall, color=INK2, linewidth=1.2, linestyle="--")
    bottom.set_ylabel("1년당 사고율")
    bottom.set_xlabel(col)
axes[0][0].legend(frameon=False)
axes[1][0].text(0.99, 0.95, f"회색 점선 = 전체 {overall:.3f}", transform=axes[1][0].transAxes,
                ha="right", va="top", fontsize=9, color=INK2)
fig.suptitle("숫자 변수: 위 = train/test가 비슷하게 나뉘었나? · 아래 = 구간별 1년당 사고율 (train)",
             x=0.01, ha="left", color=INK)
save_and_show(fig, "freq_viz_2_numeric.png")


# ---------------------------------------------------------------- 3. 범주 변수: 사고 수와 1년당 사고율 (5단계)
def level_order(d, col):
    if col == "DrivAgeBin":
        return sorted(d[col].unique(), key=lambda s: float(s.strip("(]").split(",")[0]))
    if col == "VehPower":
        return sorted(d[col].unique(), key=int)
    return d.groupby(col)["ClaimNb"].sum().sort_values(ascending=False).index.tolist()   # 사고 많은 순


# Region, VehBrand는 병합 "전" 범주로 그려서 어떤 게 왜 Other로 갔는지 보여 줌
cat_cols = [("DrivAgeBin", "운전자 나이 구간", train, None),
            ("VehPower", "차 힘 등급 (12 이상은 12)", train, None),
            ("VehGas", "연료", train, None),
            ("VehBrand", "브랜드 (회색 = Other로 합침)", train_raw, params["keep_brand"]),
            ("Region", "지역 (회색 = Other로 합침)", train_raw, params["keep_region"])]

fig, axes = plt.subplots(len(cat_cols), 2, figsize=(14, 3.2 * len(cat_cols)))
for row, (col, title, d, keep) in zip(axes, cat_cols):
    order = level_order(d, col)
    g = d.groupby(col)[["ClaimNb", "Exposure"]].sum().reindex(order)
    x = np.arange(len(order))
    rot = 45 if col == "Region" else 0
    ha = "right" if rot else "center"
    colors = [BLUE if keep is None or lv in keep else GRAY for lv in order]

    # 왼쪽: 사고 수 (행 수가 아님!) — 희귀 범주 판단 기준
    row[0].bar(x, g["ClaimNb"], color=colors, width=0.7)
    if keep is not None:
        row[0].axhline(200, color=ORANGE, linewidth=1.2, linestyle="--")
        row[0].text(0.99, 0.95, "주황 점선 = Other 기준 사고 200건", transform=row[0].transAxes,
                    ha="right", va="top", fontsize=9, color=INK2)
    row[0].set_title(f"{title}: train 사고 수", loc="left")
    row[0].set_xticks(x, order, rotation=rot, ha=ha)

    # 오른쪽: 1년당 사고율 (전체 평균선과 비교)
    row[1].bar(x, g["ClaimNb"] / g["Exposure"], color=colors, width=0.7)
    row[1].axhline(overall, color=INK2, linewidth=1.2, linestyle="--")
    row[1].set_title(f"{title}: 1년당 사고율", loc="left")
    row[1].set_xticks(x, order, rotation=rot, ha=ha)

axes[0][1].text(0.99, 0.95, f"회색 점선 = 전체 {overall:.3f}", transform=axes[0][1].transAxes,
                ha="right", va="top", fontsize=9, color=INK2)
save_and_show(fig, "freq_viz_3_categorical.png")

print("저장: freq_viz_1_target.png, freq_viz_2_numeric.png, freq_viz_3_categorical.png")

[시각화] Colab에 한글 폰트가 없어서 나눔 폰트를 설치하는 중... (10~30초)

[시각화] 사용 폰트: NanumGothic
저장: freq_viz_1_target.png, freq_viz_2_numeric.png, freq_viz_3_categorical.png
